# spi_nb_bronze_construction

**Purpose.** Bronze ingestion of the MITMA construction XLS files into the
Lakehouse, for the Construction source (source 4 of `spi_pl_bronze`, per
ADR-007).

**Inputs.** Four legacy XLS files from
`https://apps.fomento.gob.es/BoletinOnline/sedal/`: `01401400.XLS`,
`01401600.XLS` (contracting agent `estado`), `01402600.XLS`, `01402800.XLS`
(contracting agent `entes`). No runtime parameters.

**Outputs.** Delta table `spi_bronze_construction_raw` in the default
Lakehouse, written with `mode="overwrite"`. The table holds the raw grid of
all four files appended (ADR-013): positional columns `col_01` … `col_14`,
null where a file has fewer columns. Every row carries `contracting_agent`,
`is_continuation`, `_ingestion_timestamp` and `_source_file`; all columns are
stored as string, per the Bronze layer principle.

**Dependencies.** `pandas`, `requests`, `xlrd`, `deltalake`. Runs on the
Fabric **Python** runtime (no Spark session), per ADR-012.

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>}`. On failure, the exception propagates uncaught; a
chained Script activity reads the Notebook activity's status, per the
2026-09-30 addendum to ADR-009.

**Last updated.** 2026-10-01.

In [ ]:
%pip install xlrd==2.0.1
from datetime import datetime, timezone
import json
import logging

import pandas as pd
import requests
import xlrd
from deltalake import write_deltalake

# ────────────────────────────────────────────────────────────────────────────
# Configure environment variables
# ────────────────────────────────────────────────────────────────────────────

URL_BASE = "https://apps.fomento.gob.es/BoletinOnline/sedal/"
FILES = [
    {"file_id": "01401400.XLS", "agent": "estado",  "is_continuation": False},
    {"file_id": "01401600.XLS", "agent": "estado",  "is_continuation": True},
    {"file_id": "01402600.XLS", "agent": "entes",   "is_continuation": False},
    {"file_id": "01402800.XLS", "agent": "entes",   "is_continuation": True},
]

TIMEOUT = 30

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("cons_xls_bronze_load")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function download_file
# ────────────────────────────────────────────────────────────────────────────

def download_file(file_id: str) -> bytes:
    """
    Build the MiTMA URL with the constant and specified parameter.
    Download the file into memory.
    Args:
        file_id (str): The value of the file_id parameter for the URL
    Returns:
        bytes: Content of the downloaded XLS file.
    """
    xls_path = f"{URL_BASE}{file_id}"
    log.info("Downloading: %s", file_id)
    response = requests.get(xls_path, timeout=TIMEOUT)
    response.raise_for_status() #if failed (e.g. 4xx), the process die.
    log.info(
        "File downloaded: %s  |  Bytes: %s",
        file_id, len(response.content),
    )
    return response.content

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function cell_to_str
# ────────────────────────────────────────────────────────────────────────────

def cell_to_str(value) -> str:
    """Convert a xlrd cell to string. (Bronze layer)"""
    if isinstance(value, float):
        if value == int(value):
            return str(int(value))
        else:
            return str(value)
    else:
        return str(value)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function parse_construction_xls
# ────────────────────────────────────────────────────────────────────────────

def parse_construction_xls(
        content: bytes,
        file_id: str,
        contracting_agent: str,
        is_continuation: bool
) -> pd.DataFrame:
    """
    Read the XLS file content and return a DataFrame
    Args:
        content (bytes): Content of the downloaded XLS file.
        file_id (str): Name of the source file (FILES file_id).
        contracting_agent (str): Metadata from the constant FILES (agent)
        is_continuation (bool): Metadata from the constant FILES (is_continuation)
    Returns:
        pd.DataFrame: A Dataframe with data and 4 lineage columns
        (_ingestion_timestamp & _source_file, contracting_agent, is_continuation)
    """
    wb = xlrd.open_workbook(file_contents=content)
    sh = wb.sheet_by_index(0)
    all_rows = []
    for r in range(sh.nrows):
        row = []
        for c in range(sh.ncols):
            row.append(cell_to_str(sh.cell_value(r, c)))
        all_rows.append(row)
    col_names = [f"col_{i:02d}" for i in range(1, sh.ncols + 1)]
    df = pd.DataFrame(all_rows, columns=col_names)
    df["contracting_agent"] = contracting_agent
    df["is_continuation"] = str(is_continuation) # Stored as string by the principle of bronze layer
    df["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
    df["_source_file"] = file_id
    log.info(
        "Parsed: %s  |  Rows: %s  |  Cols: %s",
        file_id, df.shape[0], df.shape[1],
    )
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> int:
    """
    Main function that executes the entire process Data loading from MiTMA URL.
    Returns:
        int: Number of rows written to the Bronze table.
    """
    dfs = []
    for f in FILES:
        content = download_file(f["file_id"])
        df = parse_construction_xls(content, f["file_id"], f["agent"], f["is_continuation"])
        dfs.append(df)

    # Files have 12-14 columns; missing positional columns stay null (ADR-013)
    df = pd.concat(dfs, ignore_index=True)
    table_uri = (
        "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
        "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables/spi_bronze_construction_raw"
    )
    storage_options = {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }
    write_deltalake(table_uri, df, mode="overwrite", storage_options=storage_options)
    log.info(f"Data stored in spi_bronze_construction_raw: {len(df)} rows")
    return len(df)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to scr_log_construction (ADR-009).

rows_written = main()
notebookutils.notebook.exit(json.dumps({"rows_written": rows_written}))